# Praktikum Pengolahan Citra Digital: Sub-CPMK 5
## Meredam Derau dengan Penyaringan Spasial

Setelah praktikum ini, mahasiswa dapat membedakan derau Gaussian dan *salt-and-pepper*, menerapkan mean, Gaussian, median, serta bilateral, lalu memilih hasil berdasarkan detail citra, MSE, dan PSNR.

**Pola kerja:** prediksi, jalankan, amati, jelaskan, lalu ubah satu parameter.

## Aturan praktikum

1. Semua filter menerima input berderau yang sama. Derau dibuat sekali dengan *seed* tetap.
2. Prediksi ditulis sebelum hasil filter ditampilkan.
3. Periksa area halus, tepi, dan tekstur sebelum membaca MSE/PSNR.
4. Mean, Gaussian, dan median memakai ukuran 5 x 5. Bilateral memakai `d=9`, `sigmaColor=75`, `sigmaSpace=75`. Catat parameter ini ketika membandingkan hasil.
5. Citra contoh tersedia melalui scikit-image; tidak perlu mengunggah gambar ke Colab.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import data
from skimage.metrics import mean_squared_error, peak_signal_noise_ratio

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 11
print("OpenCV:", cv2.__version__)
print("NumPy :", np.__version__)

In [ ]:
# Kedua citra contoh tersedia melalui scikit-image; tidak perlu mengunggah aset.
demo_bersih = data.camera().astype(np.uint8)
latihan_bersih = cv2.resize(
    data.coins().astype(np.uint8), (512, 512), interpolation=cv2.INTER_CUBIC
)

print("Demo   :", demo_bersih.shape, demo_bersih.dtype)
print("Latihan:", latihan_bersih.shape, latihan_bersih.dtype)

In [ ]:
def tambah_derau_gaussian(image, sigma=18, seed=2026):
    rng = np.random.default_rng(seed)
    noise = rng.normal(0, sigma, image.shape)
    return np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)

def tambah_salt_pepper(image, amount=0.05, seed=2027):
    rng = np.random.default_rng(seed)
    result = image.copy()
    selector = rng.random(image.shape)
    result[selector < amount / 2] = 0
    result[(selector >= amount / 2) & (selector < amount)] = 255
    return result

def tampilkan(image, title="Citra", ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

def bandingkan(images, titles, figsize=(15, 4)):
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, image, title in zip(axes, images, titles):
        tampilkan(image, title, ax)
    plt.tight_layout()
    plt.show()

def terapkan_empat_filter(image, ksize=5):
    return {
        "Mean": cv2.blur(image, (ksize, ksize)),
        "Gaussian": cv2.GaussianBlur(image, (ksize, ksize), sigmaX=1.2, sigmaY=1.2),
        "Median": cv2.medianBlur(image, ksize),
        "Bilateral": cv2.bilateralFilter(image, d=9, sigmaColor=75, sigmaSpace=75),
    }

def tampilkan_roi(images, titles, roi, figsize=(13, 4)):
    y1, y2, x1, x2 = roi
    crops = [image[y1:y2, x1:x2] for image in images]
    bandingkan(crops, titles, figsize=figsize)

def tabel_metrik(clean, noisy, filtered):
    rows = [("Input berderau", noisy)] + list(filtered.items())
    print(f"{'Hasil':16s} | {'MSE':>9s} | {'PSNR (dB)':>9s}")
    print("-" * 42)
    for name, image in rows:
        mse = mean_squared_error(clean, image)
        psnr = peak_signal_noise_ratio(clean, image, data_range=255)
        print(f"{name:16s} | {mse:9.2f} | {psnr:9.2f}")

## 1. Citra bersih dan dua jenis derau

Derau Gaussian menambahkan perubahan acak kecil pada banyak piksel. *Salt-and-pepper* mengganti sebagian piksel dengan nilai ekstrem 0 atau 255.

**Prediksi:** pada jenis derau mana piksel pencilan lebih mudah dikenali?

In [ ]:
demo_gaussian = tambah_derau_gaussian(demo_bersih, sigma=18, seed=2026)
demo_sp = tambah_salt_pepper(demo_bersih, amount=0.05, seed=2027)

bandingkan(
    [demo_bersih, demo_gaussian, demo_sp],
    ["Bersih", "Derau Gaussian", "Salt-and-pepper"],
    figsize=(12, 4),
)

## 2. Lingkungan lokal dan satu perhitungan piksel

Perhatikan lingkungan 3 x 3 berikut. Delapan piksel bernilai 100 dan piksel pusat bernilai 255.

- *Mean*: `(8 x 100 + 255) / 9 = 117,22`
- Median: nilai tengah setelah diurutkan adalah `100`

*Mean* masih dipengaruhi nilai ekstrem. Median mengabaikan pencilan selama mayoritas tetangga masih mewakili struktur lokal.

In [ ]:
lingkungan = np.array([
    [100, 100, 100],
    [100, 255, 100],
    [100, 100, 100],
], dtype=np.uint8)

print("Mean   :", lingkungan.mean())
print("Median :", np.median(lingkungan))

## 3. Kernel 2D dan penanganan batas

Kernel mean 3 x 3 mempunyai sembilan bobot `1/9`. Jalankan `cv2.filter2D` dan `cv2.blur` dengan mode batas yang sama, lalu bandingkan hasilnya. `filter2D` memakai korelasi tanpa membalik kernel; untuk kernel mean yang simetris, hasilnya sama dengan konvolusi.

Pada citra 2 x 2 `[[50, 80], [60, 90]]`, lingkungan piksel pojok kiri atas dengan *replicate padding* menjadi `[[50, 50, 80], [50, 50, 80], [60, 60, 90]]`. Mean-nya `570/9 = 63,33`, yang disimpan sebagai `63` pada citra `uint8`.

Amati constant 0, replicate, dan reflect 101 pada potongan citra yang sama. Perbedaan terlihat di dekat tepi.

In [ ]:
kernel_mean = np.ones((3, 3), dtype=np.float32) / 9
hasil_filter2d = cv2.filter2D(demo_gaussian, -1, kernel_mean,
                              borderType=cv2.BORDER_REFLECT_101)
hasil_blur = cv2.blur(demo_gaussian, (3, 3),
                     borderType=cv2.BORDER_REFLECT_101)
selisih_maksimum = np.abs(
    hasil_filter2d.astype(np.int16) - hasil_blur.astype(np.int16)
).max()
print("Jumlah bobot kernel:", kernel_mean.sum())
print("Selisih piksel maksimum filter2D dan blur:", selisih_maksimum)

pojok_asli = np.array([[50, 80], [60, 90]], dtype=np.uint8)
lingkungan_pojok = np.pad(pojok_asli, 1, mode="edge")[:3, :3]
hasil_manual = lingkungan_pojok.mean()
hasil_opencv = cv2.filter2D(pojok_asli, -1, kernel_mean,
                           borderType=cv2.BORDER_REPLICATE)[0, 0]
print("Lingkungan pojok dengan replicate:\n", lingkungan_pojok)
print(f"Mean manual: {hasil_manual:.2f}; hasil uint8 OpenCV: {hasil_opencv}")

potongan = demo_bersih[40:190, 20:210]
kernel_batas = np.ones((9, 9), dtype=np.float32) / 81
mode_batas = {
    "Constant 0": cv2.BORDER_CONSTANT,
    "Replicate": cv2.BORDER_REPLICATE,
    "Reflect 101": cv2.BORDER_REFLECT_101,
}
hasil_batas = {
    nama: cv2.filter2D(potongan, -1, kernel_batas, borderType=mode)
    for nama, mode in mode_batas.items()
}
bandingkan([potongan] + list(hasil_batas.values()),
           ["Asli"] + list(hasil_batas), figsize=(15, 4))

## 4. Empat filter

- `cv2.blur`: seluruh tetangga memiliki bobot sama.
- `cv2.GaussianBlur`: tetangga dekat pusat memiliki bobot lebih besar.
- `cv2.medianBlur`: nilai lingkungan diurutkan dan nilai tengah dipilih.
- `cv2.bilateralFilter`: tetangga dekat dengan intensitas serupa lebih berpengaruh; tepi dapat lebih terjaga, tetapi proses lebih lambat.

Bilateral memakai diameter 9, sementara tiga filter lain memakai ukuran 5 x 5. Perbandingan ini menguji keluaran pada parameter yang dinyatakan.

### Prediksi sebelum hasil demonstrasi

Filter mana yang menurut Anda paling sesuai untuk derau Gaussian? Bagaimana untuk *salt-and-pepper*? Sebutkan alasan sebelum menjalankan sel berikut.

In [ ]:
hasil_gaussian = terapkan_empat_filter(demo_gaussian, ksize=5)
hasil_sp = terapkan_empat_filter(demo_sp, ksize=5)

bandingkan(
    [demo_gaussian] + list(hasil_gaussian.values()),
    ["Input Gaussian"] + list(hasil_gaussian.keys()),
    figsize=(18, 4),
)

bandingkan(
    [demo_sp] + list(hasil_sp.values()),
    ["Input S&P"] + list(hasil_sp.keys()),
    figsize=(18, 4),
)

## 5. Bukti pada wilayah yang sama

Gambar utuh dapat menyembunyikan perbedaan kecil. Bandingkan area yang sama untuk melihat derau tersisa, kondisi tepi, dan tekstur.

In [ ]:
roi_halus = (15, 85, 15, 125)
roi_tepi = (145, 255, 210, 330)
roi_tekstur = (300, 440, 220, 365)

print("ROI area halus")
tampilkan_roi(
    [demo_sp] + list(hasil_sp.values()),
    ["Input"] + list(hasil_sp),
    roi_halus,
    figsize=(18, 4),
)

print("ROI tepi")
tampilkan_roi(
    [demo_sp] + list(hasil_sp.values()),
    ["Input"] + list(hasil_sp),
    roi_tepi,
    figsize=(18, 4),
)

print("ROI tekstur")
tampilkan_roi(
    [demo_sp] + list(hasil_sp.values()),
    ["Input"] + list(hasil_sp),
    roi_tekstur,
    figsize=(18, 4),
)

## 6. Ukuran lingkungan

Ukuran lebih besar menggunakan lebih banyak tetangga. Derau dapat berkurang, tetapi garis tipis dan tekstur juga lebih mudah hilang.

**Prediksi:** ukuran mana yang paling banyak menghilangkan derau? Ukuran mana yang berisiko menghapus detail?

In [ ]:
ukuran_dicoba = [3, 5, 9]
hasil_ukuran = [cv2.medianBlur(demo_sp, k) for k in ukuran_dicoba]
bandingkan(
    [demo_sp] + hasil_ukuran,
    ["Input S&P"] + [f"Median {k} x {k}" for k in ukuran_dicoba],
    figsize=(15, 4),
)

## 7. MSE dan PSNR pada citra demo

Baca metrik setelah melihat citra. MSE lebih kecil dan PSNR lebih besar berarti galat piksel terhadap citra bersih lebih kecil. Metrik ini memerlukan citra bersih sebagai referensi dan tidak menjamin tepi atau tekstur tampak paling baik.

In [ ]:
print("Gaussian")
tabel_metrik(demo_bersih, demo_gaussian, hasil_gaussian)
print("\nSalt-and-pepper")
tabel_metrik(demo_bersih, demo_sp, hasil_sp)

## 8. Latihan wajib: ulangi demonstrasi pada citra koin

Gunakan `latihan_bersih`. Jalankan sel dari atas ke bawah. Buat dua jenis derau dengan *seed* yang sudah ditentukan, tulis prediksi, lalu bandingkan keempat filter. Fungsi bantu sudah tersedia.

In [ ]:
# Gunakan citra latihan yang sama untuk kedua jenis derau.
latihan_gaussian = tambah_derau_gaussian(latihan_bersih, sigma=18, seed=3026)
latihan_sp = tambah_salt_pepper(latihan_bersih, amount=0.05, seed=3027)
bandingkan([latihan_bersih, latihan_gaussian, latihan_sp],
           ["Bersih", "Gaussian", "Salt-and-pepper"], figsize=(12, 4))

### Prediksi mahasiswa: isi sebelum menjalankan empat filter

- Derau Gaussian: saya memprediksi filter `...` karena `...`.
- *Salt-and-pepper*: saya memprediksi filter `...` karena `...`.

In [ ]:
# Ketiga filter pertama memakai ukuran 5 x 5. Bilateral memakai d=9.
# Semua metode menerima input berderau yang sama, tanpa pemrosesan bertingkat.
ukuran_filter = 5
hasil_latihan_gaussian = terapkan_empat_filter(latihan_gaussian, ukuran_filter)
hasil_latihan_sp = terapkan_empat_filter(latihan_sp, ukuran_filter)

bandingkan(
    [latihan_gaussian] + list(hasil_latihan_gaussian.values()),
    ["Input Gaussian"] + list(hasil_latihan_gaussian.keys()),
    figsize=(18, 4),
)

bandingkan(
    [latihan_sp] + list(hasil_latihan_sp.values()),
    ["Input S&P"] + list(hasil_latihan_sp.keys()),
    figsize=(18, 4),
)

### Periksa area koin yang sama

Potongan berikut memuat tepi koin, bagian koin, dan latar. Bandingkan setiap hasil dari area yang identik.

In [ ]:
roi_koin = (255, 395, 265, 405)
tampilkan_roi([latihan_sp] + list(hasil_latihan_sp.values()),
             ["Input"] + list(hasil_latihan_sp), roi_koin, figsize=(18, 4))

### Uji ukuran median pada salt-and-pepper

Jalankan ukuran 3, 5, dan 9. Pilih ukuran yang seimbang dengan menyebut bintik yang hilang dan detail koin yang berubah.

In [ ]:
# Ulangi eksperimen ukuran median pada salt-and-pepper.
ukuran_median = [3, 5, 9]
hasil_median_ukuran = [cv2.medianBlur(latihan_sp, k) for k in ukuran_median]
bandingkan(
    [latihan_sp] + hasil_median_ukuran,
    ["Input S&P"] + [f"Median {k} x {k}" for k in ukuran_median],
    figsize=(15, 4),
)

### Tabel pengamatan dan pilihan visual sementara

| Derau | Filter | Derau tersisa | Kondisi tepi | Kondisi tekstur | Keputusan |
|---|---|---|---|---|---|
| Gaussian | Mean | ... | ... | ... | ... |
| Gaussian | Gaussian | ... | ... | ... | ... |
| Gaussian | Median | ... | ... | ... | ... |
| Gaussian | Bilateral | ... | ... | ... | ... |
| Salt-and-pepper | Mean | ... | ... | ... | ... |
| Salt-and-pepper | Gaussian | ... | ... | ... | ... |
| Salt-and-pepper | Median | ... | ... | ... | ... |
| Salt-and-pepper | Bilateral | ... | ... | ... | ... |

**Pilihan sebelum membaca metrik:** Gaussian `...`; *salt-and-pepper* `...`. Bukti visual saya terlihat pada area `...`.

## 9. MSE dan PSNR pada citra latihan

Bandingkan nilai dengan pilihan visual sementara. MSE lebih kecil dan PSNR lebih besar menunjukkan galat piksel yang lebih kecil terhadap citra bersih. Jelaskan bila angka dan kesan visual Anda berbeda.

In [ ]:
# Baca angka setelah menulis pilihan berdasarkan tampilan citra.
print("Gaussian")
tabel_metrik(latihan_bersih, latihan_gaussian, hasil_latihan_gaussian)
print("\nSalt-and-pepper")
tabel_metrik(latihan_bersih, latihan_sp, hasil_latihan_sp)

## 10. Kesimpulan dan pengumpulan

**Kesimpulan 3-5 kalimat:** “Untuk derau ..., saya memilih filter ... dengan parameter ... karena pada area ... terlihat .... MSE/PSNR hasilnya adalah .... Efek samping yang masih terlihat ialah ....”

Kumpulkan notebook yang sudah dijalankan. Pastikan prediksi, dua perbandingan empat filter, ROI koin, uji median 3/5/9, tabel delapan baris, metrik, dan kesimpulan sudah terlihat.

**Opsional setelah selesai:** coba citra pribadi. Tanpa citra bersih asli, MSE/PSNR tidak dapat dihitung dengan cara yang sama.